# Notebook 21 — Adaptive Threshold Gating

**Repo:** `int_serialization_benchmark-rml`  
**Layer:** `rml_extension/notebooks/`

Notebook 20 used fixed CGCS-style macro-route gates:

- accept threshold = `0.70`
- watch threshold = `0.50`

Notebook 21 makes those thresholds adaptive.

Constraint view:
> macro-routing gates should tighten under instability and relax under stable coherent plateaus.

## Goals

1. Load Notebook 20 constraint-gated macro routing outputs.
2. Estimate rolling route pressure and CGCS distribution.
3. Build adaptive accept/watch thresholds.
4. Compare fixed vs adaptive gates.
5. Detect early decompression candidates.
6. Export CSV, JSON, Markdown report, and PNG figures.

This notebook generates a full report matching the previous repo format:

- generated output links,
- summary table,
- transition probability tables,
- interpretation,
- next step.

In [ ]:
from pathlib import Path
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd()
candidates = [
    cwd,
    cwd.parent,
    cwd.parent.parent,
    Path("/content/int_serialization_benchmark-rml"),
    Path("/content"),
]

REPO_ROOT = None
for c in candidates:
    if (c / "rml_extension").exists() or (c / "configs").exists():
        REPO_ROOT = c
        break

if REPO_ROOT is None:
    REPO_ROOT = cwd

RML_ROOT = REPO_ROOT / "rml_extension" if (REPO_ROOT / "rml_extension").exists() else REPO_ROOT

RESULTS_DIR = RML_ROOT / "results"
FIGURES_DIR = RML_ROOT / "figures"
REPORTS_DIR = RML_ROOT / "reports"

for d in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("RML_ROOT:", RML_ROOT)

## Load Notebook 20 outputs

Uses:

```text
results/notebook20_constraint_gated_macro_routing.csv
```

If that file is not available, a fallback synthetic stream is created so the notebook runs end-to-end.

In [ ]:
input_path = RESULTS_DIR / "notebook20_constraint_gated_macro_routing.csv"

if input_path.exists():
    df = pd.read_csv(input_path)
    print("Loaded:", input_path)
else:
    print("Notebook 20 output not found; creating fallback synthetic data.")
    rng = np.random.default_rng(21)
    n = 240
    macro_routes = [f"macro_{i}" for i in range(5)]
    route = []
    for i in range(n):
        if 100 <= i <= 145:
            route.append("macro_0")
        else:
            route.append(rng.choice(macro_routes))

    score = np.clip(rng.normal(0.53, 0.11, n), 0, 1)
    score[100:146] = np.clip(rng.normal(0.68, 0.05, 46), 0, 1)

    df = pd.DataFrame({
        "window_id": np.arange(n),
        "macro_route": route,
        "macro_cgcs_score": score,
        "macro_switch_rate": np.clip(rng.normal(0.62, 0.24, n), 0, 1),
        "child_switch_rate": np.clip(rng.normal(0.64, 0.22, n), 0, 1),
        "policy_switch_rate": np.clip(rng.normal(0.60, 0.25, n), 0, 1),
        "macro_compression_residual": np.clip(rng.normal(0.10, 0.07, n), 0, 1),
        "compressed_route_stability": np.clip(rng.normal(0.45, 0.20, n), 0, 1),
    })
    df.loc[100:145, ["macro_switch_rate", "child_switch_rate", "policy_switch_rate"]] = 0.0
    df.loc[100:145, "compressed_route_stability"] = np.clip(rng.normal(0.95, 0.04, 46), 0, 1)

df = df.sort_values("window_id").reset_index(drop=True)

required_cols = [
    "window_id",
    "macro_route",
    "macro_cgcs_score",
    "macro_switch_rate",
    "child_switch_rate",
    "policy_switch_rate",
    "macro_compression_residual",
    "compressed_route_stability",
]

for c in required_cols:
    if c not in df.columns:
        if c == "window_id":
            df[c] = np.arange(len(df))
        elif c == "macro_route":
            df[c] = "macro_unknown"
        else:
            df[c] = 0.5

numeric_cols = [
    "macro_cgcs_score",
    "macro_switch_rate",
    "child_switch_rate",
    "policy_switch_rate",
    "macro_compression_residual",
    "compressed_route_stability",
]
for c in numeric_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.5).clip(0, 1)

df.head()

## Rolling instability estimates

Adaptive thresholds use local context:

- rolling route pressure,
- rolling CGCS mean,
- rolling CGCS volatility,
- rolling residual pressure,
- rolling compressed-route stability.

In [ ]:
roll = 21

df["macro_route_changed"] = df["macro_route"].ne(df["macro_route"].shift(1)).fillna(False)
df["rolling_route_switch"] = df["macro_route_changed"].rolling(roll, min_periods=1).mean()

df["rolling_pressure"] = (
    0.40 * df["macro_switch_rate"].rolling(roll, min_periods=1).mean()
    + 0.25 * df["child_switch_rate"].rolling(roll, min_periods=1).mean()
    + 0.25 * df["policy_switch_rate"].rolling(roll, min_periods=1).mean()
    + 0.10 * df["rolling_route_switch"]
).clip(0, 1)

df["rolling_cgcs_mean"] = df["macro_cgcs_score"].rolling(roll, min_periods=1).mean()
df["rolling_cgcs_std"] = df["macro_cgcs_score"].rolling(roll, min_periods=2).std().fillna(0.0).clip(0, 1)
df["rolling_residual"] = df["macro_compression_residual"].rolling(roll, min_periods=1).mean().clip(0, 1)
df["rolling_stability"] = df["compressed_route_stability"].rolling(roll, min_periods=1).mean().clip(0, 1)

df[[
    "window_id",
    "macro_cgcs_score",
    "rolling_pressure",
    "rolling_cgcs_mean",
    "rolling_cgcs_std",
    "rolling_residual",
    "rolling_stability",
]].head()

## Adaptive threshold rule

Thresholds tighten when:

- route pressure rises,
- CGCS volatility rises,
- residual pressure rises.

Thresholds relax when compressed-route stability rises.

In [ ]:
fixed_accept = 0.70
fixed_watch = 0.50

df["adaptive_accept_threshold"] = (
    fixed_accept
    + 0.10 * df["rolling_pressure"]
    + 0.08 * df["rolling_cgcs_std"]
    + 0.06 * df["rolling_residual"]
    - 0.08 * df["rolling_stability"]
).clip(0.58, 0.86)

df["adaptive_watch_threshold"] = (
    fixed_watch
    + 0.07 * df["rolling_pressure"]
    + 0.05 * df["rolling_cgcs_std"]
    + 0.04 * df["rolling_residual"]
    - 0.06 * df["rolling_stability"]
).clip(0.38, 0.68)

df["adaptive_accept_threshold"] = np.maximum(
    df["adaptive_accept_threshold"],
    df["adaptive_watch_threshold"] + 0.10
).clip(0, 1)

def fixed_gate(x):
    if x >= fixed_accept:
        return "accepted"
    if x >= fixed_watch:
        return "watch"
    return "fallback"

def adaptive_gate(row):
    x = row["macro_cgcs_score"]
    if x >= row["adaptive_accept_threshold"]:
        return "accepted"
    if x >= row["adaptive_watch_threshold"]:
        return "watch"
    return "fallback"

df["fixed_gate"] = df["macro_cgcs_score"].apply(fixed_gate)
df["adaptive_gate"] = df.apply(adaptive_gate, axis=1)

df[[
    "window_id",
    "macro_cgcs_score",
    "adaptive_watch_threshold",
    "adaptive_accept_threshold",
    "fixed_gate",
    "adaptive_gate",
]].head()

## Adaptive decompression policy

Decompress early when:

- adaptive gate falls to fallback,
- CGCS score approaches the watch threshold from above,
- route pressure rises while CGCS weakens.

In [ ]:
df["cgcs_slope"] = df["macro_cgcs_score"].diff().rolling(5, min_periods=1).mean().fillna(0.0)
df["pressure_slope"] = df["rolling_pressure"].diff().rolling(5, min_periods=1).mean().fillna(0.0)

df["adaptive_margin_to_watch"] = df["macro_cgcs_score"] - df["adaptive_watch_threshold"]
df["adaptive_margin_to_accept"] = df["macro_cgcs_score"] - df["adaptive_accept_threshold"]

df["early_decompression_candidate"] = (
    df["adaptive_gate"].eq("fallback")
    | ((df["adaptive_margin_to_watch"] < 0.05) & (df["pressure_slope"] > 0))
    | ((df["cgcs_slope"] < -0.015) & (df["rolling_pressure"] > df["rolling_pressure"].median()))
)

df["adaptive_gated_route"] = np.where(
    df["adaptive_gate"].eq("fallback"),
    "adaptive_fallback",
    df["macro_route"],
)

df["fixed_gated_route"] = np.where(
    df["fixed_gate"].eq("fallback"),
    "fixed_fallback",
    df["macro_route"],
)

df[[
    "window_id",
    "adaptive_gate",
    "adaptive_margin_to_watch",
    "early_decompression_candidate",
]].head()

## Compare fixed vs adaptive gates

In [ ]:
def rolling_switch(series, window=21):
    return series.ne(series.shift(1)).fillna(False).rolling(window, min_periods=1).mean()

df["fixed_gated_switch_rate"] = rolling_switch(df["fixed_gated_route"], roll)
df["adaptive_gated_switch_rate"] = rolling_switch(df["adaptive_gated_route"], roll)

df["fixed_stability_score"] = (
    1.0
    - 0.65 * df["fixed_gated_switch_rate"]
    - 0.25 * df["fixed_gate"].eq("fallback").astype(float)
    - 0.10 * (df["macro_compression_residual"] > df["macro_compression_residual"].quantile(0.80)).astype(float)
).clip(0, 1)

df["adaptive_stability_score"] = (
    1.0
    - 0.65 * df["adaptive_gated_switch_rate"]
    - 0.20 * df["adaptive_gate"].eq("fallback").astype(float)
    - 0.15 * df["early_decompression_candidate"].astype(float)
).clip(0, 1)

comparison = {
    "windows": int(len(df)),
    "fixed_accepted": int((df["fixed_gate"] == "accepted").sum()),
    "adaptive_accepted": int((df["adaptive_gate"] == "accepted").sum()),
    "fixed_watch": int((df["fixed_gate"] == "watch").sum()),
    "adaptive_watch": int((df["adaptive_gate"] == "watch").sum()),
    "fixed_fallback": int((df["fixed_gate"] == "fallback").sum()),
    "adaptive_fallback": int((df["adaptive_gate"] == "fallback").sum()),
    "early_decompression_candidates": int(df["early_decompression_candidate"].sum()),
    "mean_fixed_switch_rate": float(df["fixed_gated_switch_rate"].mean()),
    "mean_adaptive_switch_rate": float(df["adaptive_gated_switch_rate"].mean()),
    "mean_fixed_stability_score": float(df["fixed_stability_score"].mean()),
    "mean_adaptive_stability_score": float(df["adaptive_stability_score"].mean()),
}

pd.DataFrame([comparison])

## Gate transition matrices

In [ ]:
gate_labels = ["accepted", "watch", "fallback"]

def transition_matrix(series):
    mat = pd.DataFrame(0.0, index=gate_labels, columns=gate_labels)
    vals = list(series)
    for a, b in zip(vals[:-1], vals[1:]):
        mat.loc[a, b] += 1
    return mat.div(mat.sum(axis=1).replace(0, np.nan), axis=0).fillna(0.0)

fixed_transition = transition_matrix(df["fixed_gate"])
adaptive_transition = transition_matrix(df["adaptive_gate"])

adaptive_transition

## Export result tables

In [ ]:
csv_path = RESULTS_DIR / "notebook21_adaptive_threshold_gating.csv"
json_path = RESULTS_DIR / "notebook21_adaptive_threshold_gating.json"
comparison_path = RESULTS_DIR / "notebook21_fixed_vs_adaptive_summary.json"
fixed_transition_path = RESULTS_DIR / "notebook21_fixed_gate_transition_matrix.csv"
adaptive_transition_path = RESULTS_DIR / "notebook21_adaptive_gate_transition_matrix.csv"

df.to_csv(csv_path, index=False)
df.to_json(json_path, orient="records", indent=2)
comparison_path.write_text(json.dumps(comparison, indent=2))
fixed_transition.to_csv(fixed_transition_path)
adaptive_transition.to_csv(adaptive_transition_path)

print("Saved:", csv_path)
print("Saved:", json_path)
print("Saved:", comparison_path)
print("Saved:", fixed_transition_path)
print("Saved:", adaptive_transition_path)

## Figure 1 — Adaptive threshold timeline

In [ ]:
fig_path_1 = FIGURES_DIR / "notebook21_adaptive_threshold_timeline.png"

plt.figure(figsize=(12, 4))
plt.plot(df["window_id"], df["macro_cgcs_score"], label="macro CGCS score")
plt.plot(df["window_id"], df["adaptive_accept_threshold"], linestyle="--", label="adaptive accept threshold")
plt.plot(df["window_id"], df["adaptive_watch_threshold"], linestyle="--", label="adaptive watch threshold")
plt.xlabel("Window")
plt.ylabel("Score")
plt.title("Adaptive Threshold Gating: Threshold Timeline")
plt.legend()
plt.tight_layout()
plt.savefig(fig_path_1, dpi=160)
plt.show()

print("Saved:", fig_path_1)

## Figure 2 — Fixed vs adaptive gate timeline

In [ ]:
fig_path_2 = FIGURES_DIR / "notebook21_fixed_vs_adaptive_gate_timeline.png"

gate_to_id = {"fallback": 0, "watch": 1, "accepted": 2}

plt.figure(figsize=(12, 4))
plt.step(df["window_id"], df["fixed_gate"].map(gate_to_id), where="mid", label="fixed gate")
plt.step(df["window_id"], df["adaptive_gate"].map(gate_to_id), where="mid", linestyle="--", label="adaptive gate")
plt.yticks([0, 1, 2], ["fallback", "watch", "accepted"])
plt.xlabel("Window")
plt.ylabel("Gate")
plt.title("Adaptive Threshold Gating: Fixed vs Adaptive Gate Timeline")
plt.legend()
plt.tight_layout()
plt.savefig(fig_path_2, dpi=160)
plt.show()

print("Saved:", fig_path_2)

## Figure 3 — Early decompression candidates

In [ ]:
fig_path_3 = FIGURES_DIR / "notebook21_early_decompression_candidates.png"

plt.figure(figsize=(12, 4))
plt.plot(df["window_id"], df["adaptive_margin_to_watch"], label="margin to adaptive watch threshold")
candidates = df[df["early_decompression_candidate"]]
plt.scatter(candidates["window_id"], candidates["adaptive_margin_to_watch"], s=25, label="early decompression candidate")
plt.axhline(0.0, linestyle="--", label="watch boundary")
plt.xlabel("Window")
plt.ylabel("CGCS - watch threshold")
plt.title("Adaptive Threshold Gating: Early Decompression Candidates")
plt.legend()
plt.tight_layout()
plt.savefig(fig_path_3, dpi=160)
plt.show()

print("Saved:", fig_path_3)

## Figure 4 — Fixed vs adaptive switch rates

In [ ]:
fig_path_4 = FIGURES_DIR / "notebook21_fixed_vs_adaptive_switch_rates.png"

plt.figure(figsize=(12, 4))
plt.plot(df["window_id"], df["fixed_gated_switch_rate"], label="fixed gated switch rate")
plt.plot(df["window_id"], df["adaptive_gated_switch_rate"], label="adaptive gated switch rate")
plt.xlabel("Window")
plt.ylabel("Rolling switch rate")
plt.title("Adaptive Threshold Gating: Fixed vs Adaptive Switch Rates")
plt.legend()
plt.tight_layout()
plt.savefig(fig_path_4, dpi=160)
plt.show()

print("Saved:", fig_path_4)

## Figure 5 — Fixed vs adaptive stability score

In [ ]:
fig_path_5 = FIGURES_DIR / "notebook21_fixed_vs_adaptive_stability.png"

plt.figure(figsize=(12, 4))
plt.plot(df["window_id"], df["fixed_stability_score"], label="fixed stability")
plt.plot(df["window_id"], df["adaptive_stability_score"], label="adaptive stability")
plt.xlabel("Window")
plt.ylabel("Stability score")
plt.title("Adaptive Threshold Gating: Fixed vs Adaptive Stability")
plt.legend()
plt.tight_layout()
plt.savefig(fig_path_5, dpi=160)
plt.show()

print("Saved:", fig_path_5)

## Figure 6 — Threshold pressure components

In [ ]:
fig_path_6 = FIGURES_DIR / "notebook21_threshold_pressure_components.png"

plt.figure(figsize=(12, 4))
plt.plot(df["window_id"], df["rolling_pressure"], label="rolling pressure")
plt.plot(df["window_id"], df["rolling_cgcs_std"], label="CGCS volatility")
plt.plot(df["window_id"], df["rolling_residual"], label="rolling residual")
plt.plot(df["window_id"], df["rolling_stability"], label="rolling stability")
plt.xlabel("Window")
plt.ylabel("Normalized component")
plt.title("Adaptive Threshold Gating: Pressure Components")
plt.legend()
plt.tight_layout()
plt.savefig(fig_path_6, dpi=160)
plt.show()

print("Saved:", fig_path_6)

## Figure 7 — Adaptive gate transition matrix

In [ ]:
fig_path_7 = FIGURES_DIR / "notebook21_adaptive_gate_transition_matrix.png"

plt.figure(figsize=(6, 5))
plt.imshow(adaptive_transition.values, aspect="auto", vmin=0, vmax=1)
plt.xticks(range(len(gate_labels)), gate_labels, rotation=45, ha="right")
plt.yticks(range(len(gate_labels)), gate_labels)
plt.colorbar(label="Transition probability")
plt.xlabel("Next gate")
plt.ylabel("Current gate")
plt.title("Adaptive Threshold Gating: Gate Transition Matrix")
plt.tight_layout()
plt.savefig(fig_path_7, dpi=160)
plt.show()

print("Saved:", fig_path_7)

## Figure 8 — Fixed vs adaptive gate counts

In [ ]:
fig_path_8 = FIGURES_DIR / "notebook21_fixed_vs_adaptive_gate_counts.png"

count_df = pd.DataFrame({
    "fixed": df["fixed_gate"].value_counts().reindex(gate_labels, fill_value=0),
    "adaptive": df["adaptive_gate"].value_counts().reindex(gate_labels, fill_value=0),
})

x = np.arange(len(gate_labels))
width = 0.35

plt.figure(figsize=(8, 5))
plt.bar(x - width/2, count_df["fixed"], width, label="fixed")
plt.bar(x + width/2, count_df["adaptive"], width, label="adaptive")
plt.xticks(x, gate_labels, rotation=45, ha="right")
plt.ylabel("Window count")
plt.title("Adaptive Threshold Gating: Gate Counts")
plt.legend()
plt.tight_layout()
plt.savefig(fig_path_8, dpi=160)
plt.show()

print("Saved:", fig_path_8)

## Full Markdown report

The report uses repo-relative HTML links, matching the updated Notebook 20 report style:

```md
- Result: <a href="results/...">`results/...`</a>
- Figure: <a href="/figures/...">`/figures/...`</a>
```

In [ ]:
report_path = REPORTS_DIR / "report_21_adaptive_threshold_gating.md"

lines = [
    "# Report 21 — Adaptive Threshold Gating",
    "",
    "This report replaces fixed CGCS-style macro-route gates with adaptive thresholds.",
    "",
    "Constraint view:",
    "> macro-routing gates should tighten under instability and relax under stable coherent plateaus.",
    "",
    "## Generated outputs",
    "",
    '- Adaptive threshold gating CSV: <a href="results/notebook21_adaptive_threshold_gating.csv">`results/notebook21_adaptive_threshold_gating.csv`</a>',
    '- Adaptive threshold gating JSON: <a href="results/notebook21_adaptive_threshold_gating.json">`results/notebook21_adaptive_threshold_gating.json`</a>',
    '- Fixed vs adaptive summary JSON: <a href="results/notebook21_fixed_vs_adaptive_summary.json">`results/notebook21_fixed_vs_adaptive_summary.json`</a>',
    '- Fixed gate transition matrix CSV: <a href="results/notebook21_fixed_gate_transition_matrix.csv">`results/notebook21_fixed_gate_transition_matrix.csv`</a>',
    '- Adaptive gate transition matrix CSV: <a href="results/notebook21_adaptive_gate_transition_matrix.csv">`results/notebook21_adaptive_gate_transition_matrix.csv`</a>',
    '- Figure: <a href="/figures/notebook21_adaptive_threshold_timeline.png">`/figures/notebook21_adaptive_threshold_timeline.png`</a>',
    '- Figure: <a href="/figures/notebook21_fixed_vs_adaptive_gate_timeline.png">`/figures/notebook21_fixed_vs_adaptive_gate_timeline.png`</a>',
    '- Figure: <a href="/figures/notebook21_early_decompression_candidates.png">`/figures/notebook21_early_decompression_candidates.png`</a>',
    '- Figure: <a href="/figures/notebook21_fixed_vs_adaptive_switch_rates.png">`/figures/notebook21_fixed_vs_adaptive_switch_rates.png`</a>',
    '- Figure: <a href="/figures/notebook21_fixed_vs_adaptive_stability.png">`/figures/notebook21_fixed_vs_adaptive_stability.png`</a>',
    '- Figure: <a href="/figures/notebook21_threshold_pressure_components.png">`/figures/notebook21_threshold_pressure_components.png`</a>',
    '- Figure: <a href="/figures/notebook21_adaptive_gate_transition_matrix.png">`/figures/notebook21_adaptive_gate_transition_matrix.png`</a>',
    '- Figure: <a href="/figures/notebook21_fixed_vs_adaptive_gate_counts.png">`/figures/notebook21_fixed_vs_adaptive_gate_counts.png`</a>',
    "",
    "## Summary",
    "",
    pd.DataFrame([comparison]).to_markdown(index=False),
    "",
    "## Adaptive gate transition probabilities",
    "",
    adaptive_transition.to_markdown(),
    "",
    "## Fixed gate transition probabilities",
    "",
    fixed_transition.to_markdown(),
    "",
    "## Interpretation",
    "",
    "- Adaptive gates tighten during high pressure, high volatility, and high residual windows.",
    "- Adaptive gates relax during stable compressed-route plateaus.",
    "- Early decompression candidates identify windows where macro compression may need temporary expansion.",
    "- Fixed gates are easier to interpret; adaptive gates are better for streaming pressure-sensitive routing.",
    "",
    "## Next step",
    "",
    "Notebook 22 can build predictive decompression: forecast decompression before fallback occurs.",
]

report_path.write_text("\n".join(lines))
print("Saved:", report_path)

## Optional: download output bundle in Colab

Uncomment the following cell if running in Google Colab.

In [ ]:
# OPTIONAL COLAB DOWNLOAD
#
# EXPORT_NAME = "notebook21_adaptive_threshold_gating_outputs.zip"
# export_path = RML_ROOT / EXPORT_NAME
#
# with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as zf:
#     for folder in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
#         for p in folder.glob("notebook21_*"):
#             zf.write(p, arcname=str(p.relative_to(RML_ROOT)))
#         for p in folder.glob("report_21_*"):
#             zf.write(p, arcname=str(p.relative_to(RML_ROOT)))
#
# from google.colab import files
# files.download(str(export_path))